In [21]:
#Alana Annamanthadoo

#full ML workflow for this PAH/PKU analysis:
#1.Define the problem - predict whether a PAH gene mutation is Pathogenic or Benign, connecting to why PKU is more common in populations with higher rates of consanguinity (like Turkey)
#2.Collect data  — pulled 3,420 PAH variants from ClinVar (NCBI), using chunked reading so we never had to hold the whole giant file in memory at once
#3.Clean and prepare the data - need to check the classification breakdown, then simplify down to just clear Pathogenic vs. Benign, dropping "Uncertain significance" and similar
#4.Feature engineering
#5.Split the data
#6.Choose a model 
#7.Train the model 
#8.Evaluate the model 
#9.Interpret the results 
#10.Iterate/improve 



#STEP 1,Define the problem: Predict whether a PAH gene mutation is Pathogenic or Benign, connecting to why PKU is more common in populations with higher rates of consanguinity (like Turkey)

#STEP 2 : Collect data — pulled 3,420 PAH variants from ClinVar (NCBI), using chunked reading so we never had to hold the whole giant file in memory at once
import pandas as pd

url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"

chunks_kept = []

for chunk in pd.read_csv(url, sep="\t", chunksize=100000, low_memory=False):
    pah_only = chunk[chunk["GeneSymbol"] == "PAH"]
    chunks_kept.append(pah_only)

df = pd.concat(chunks_kept)
print(df.shape)

#3420 PAH related variants before cleaning this is the "uncertain significance" which will filter out...3,420 is not the number we will actually train your model on — it's the raw total before we narrow it down to only the confidently-labeled Pathogenic/Benign ones
#3,420 PAH-related variants" means: ClinVar has 3,420 separate, distinct DNA changes recorded specifically within the PAH gene

(3420, 43)


In [22]:
df.head #shows first few rows

<bound method NDFrame.head of          #AlleleID                       Type  \
1084         15615  single nucleotide variant   
1085         15615  single nucleotide variant   
1086         15616  single nucleotide variant   
1087         15616  single nucleotide variant   
1088         15617  single nucleotide variant   
...            ...                        ...   
8981974    4962936  single nucleotide variant   
8991962    4968252  single nucleotide variant   
8991963    4968252  single nucleotide variant   
8991964    4968253  single nucleotide variant   
8991965    4968253  single nucleotide variant   

                                             Name  GeneID GeneSymbol  \
1084                 NM_000277.3(PAH):c.1315+1G>A    5053        PAH   
1085                 NM_000277.3(PAH):c.1315+1G>A    5053        PAH   
1086     NM_000277.3(PAH):c.1222C>T (p.Arg408Trp)    5053        PAH   
1087     NM_000277.3(PAH):c.1222C>T (p.Arg408Trp)    5053        PAH   
1088      NM_000277.3

In [23]:
#STEP 3 : Clean & prepare data (need to check the classification breakdown, then simplify down to just clear Pathogenic vs. Benign, dropping "Uncertain significance" and similar)
print(df["ClinicalSignificance"].value_counts(dropna=False))


#**NOTE : "Pathogenic" and "Likely pathogenic" mean essentially the same practical thing (probably harmful), and "Benign" and "Likely benign" also mean essentially the same thing (probably harmless)

#sorting wheather pathogntic or benign only (easier to understand)
df["ClinicalSignificance"].unique()

df["Label"] = df["ClinicalSignificance"].map({
    "Pathogenic": "Pathogenic",
    "Likely pathogenic": "Pathogenic",
    "Pathogenic/Likely pathogenic": "Pathogenic",
    "Benign": "Benign",
    "Likely benign": "Benign",
    "Benign/Likely benign": "Benign"
})

df_clean = df[df["Label"].notna()].copy()

print(df_clean.shape)
print(df_clean["Label"].value_counts())


ClinicalSignificance
Pathogenic                                      912
Likely pathogenic                               812
Likely benign                                   699
Uncertain significance                          667
Benign                                          120
Pathogenic/Likely pathogenic                     86
not provided                                     80
Conflicting classifications of pathogenicity     28
Benign/Likely benign                             10
-                                                 4
no classification for the single variant          2
Name: count, dtype: int64
(2639, 44)
Label
Pathogenic    1810
Benign         829
Name: count, dtype: int64


In [24]:
#Step 4: Feature Engineering
print(df_clean["Type"].value_counts())

print(df_clean["Label"].value_counts())
comparison2 = pd.crosstab(df_clean["Type"], df_clean["Label"], normalize="columns") * 100
print(comparison2.columns.tolist())

print(df_clean.columns.tolist())

#this tells us how many of 2,639 PAH mutations sit in that UTR-like region (marked by the asterisk) versus how many don't.
df_clean["Has_Asterisk"] = df_clean["Name"].str.contains(r"c\.\*")
print(df_clean["Has_Asterisk"].value_counts())

Type
single nucleotide variant    2185
Deletion                      301
Duplication                    67
Microsatellite                 32
Indel                          30
Insertion                      20
Inversion                       2
copy number loss                2
Name: count, dtype: int64
Label
Pathogenic    1810
Benign         829
Name: count, dtype: int64
['Benign', 'Pathogenic']
['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID', 'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)', 'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList', 'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession', 'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele', 'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines', 'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated', 'ReviewStatusC

In [25]:
#Step 4: Feature engineering
features = pd.get_dummies(df_clean["Type"])

model_data = features.copy()
model_data["Label"] = df_clean["Label"]

print(model_data.shape)
model_data.head()

#2,639 rows, 9 columns (8 mutation types + Label)

(2639, 9)


,Deletion,Duplication,Indel,Insertion,Inversion,Microsatellite,copy number loss,single nucleotide variant,Label
1084,False,False,False,False,False,False,False,True,Pathogenic
1085,False,False,False,False,False,False,False,True,Pathogenic
1086,False,False,False,False,False,False,False,True,Pathogenic
1087,False,False,False,False,False,False,False,True,Pathogenic
1088,False,False,False,False,False,False,False,True,Pathogenic


In [26]:
#Step 5: Split the data
from sklearn.model_selection import train_test_split

X = model_data.drop(columns=["Label"])
y = model_data["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

(2111, 8) (528, 8)


In [27]:
#step 6 & 7 :Choose a model and Train the model 
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [35]:
# STEP 8:Evaluate the model
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))


#What it actually does, in plain terms: normally, the computer tries to be "right" as often as possible overall,and since one group (harmful) is much bigger than the other (harmless) in your data, it can get away with mostly ignoring the harmless group and still score decently well, just by favoring the bigger group. class_weight="balanced" tells the computer: "no — treat getting a harmless one wrong as just as serious as getting a harmful one wrong, even though there are fewer harmless examples to learn from." This forces it to actually pay attention to the smaller group instead of ignoring it
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))


#*NOTES: Benign recall = 0.96. That means out of every single 100 truly harmless mutations, the computer
#now correctly catches about 96 of them.
#Pathogenic recall- 0.21, That means out of every 100 truly harmful mutations, the computer now only catches about 21 of them, missing the other 79



print(df_clean[["ReferenceAllele", "AlternateAllele"]].head(10))

print(df_clean["ReferenceAllele"].value_counts(dropna=False).head(10))


df_clean["Base_Change"] = df_clean["Name"].str.extract(r'([ACGT]>[ACGT])')
print(df_clean["Base_Change"].value_counts(dropna=False))


comparison = pd.crosstab(df_clean["Base_Change"], df_clean["Label"], normalize="columns") * 100
print(comparison)



base_change_features = pd.get_dummies(df_clean["Base_Change"])

model_data = pd.concat([features, base_change_features], axis=1)
model_data["Label"] = df_clean["Label"]

model_data = model_data.dropna()

print(model_data.shape)
model_data.head



X = model_data.drop(columns=["Label"])
y = model_data["Label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))


Accuracy: 0.4431818181818182
              precision    recall  f1-score   support

      Benign       0.36      0.96      0.52       166
  Pathogenic       0.91      0.21      0.34       362

    accuracy                           0.44       528
   macro avg       0.64      0.58      0.43       528
weighted avg       0.74      0.44      0.39       528

Accuracy: 0.4431818181818182
              precision    recall  f1-score   support

      Benign       0.36      0.96      0.52       166
  Pathogenic       0.91      0.21      0.34       362

    accuracy                           0.44       528
   macro avg       0.64      0.58      0.43       528
weighted avg       0.74      0.44      0.39       528

     ReferenceAllele AlternateAllele
1084              na              na
1085              na              na
1086              na              na
1087              na              na
1088              na              na
1089              na              na
1090              na         